# 03 - Tier 2 Model Training

Train and compare three deep learning architectures for network intrusion detection:

| Model | Architecture | Input Shape |
|-------|-------------|-------------|
| **DNN** | Dense layers with BatchNorm + Dropout | `(n_features,)` |
| **CNN** | 1D Convolutions + MaxPooling | `(n_features, 1)` |
| **LSTM** | Recurrent LSTM layers | `(1, n_features)` |

**Sections:**
1. Prepare data
2. Train each model
3. Compare validation accuracy
4. Plot training history
5. Confusion matrices

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from src.preprocessing.data_loader import DataLoader, CATEGORY_MAP
from src.preprocessing.preprocessor import DataPreprocessor
from src.tier2_ml_detection.models import build_dnn, build_cnn, build_lstm, get_training_callbacks
from src.tier2_ml_detection.feature_extractor import FeatureExtractor
from src.evaluation.visualizations import plot_confusion_matrix, plot_training_history
from src.utils.config import load_config, resolve_path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

print('Imports loaded.')

## 1. Prepare Data

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
except FileNotFoundError:
    print('Using synthetic data.')
    df = loader.generate_synthetic(n_samples=5000)

preprocessor = DataPreprocessor(config)
data = preprocessor.run_pipeline(df, label_type='multiclass')

n_features = data['n_features']
n_classes = data['n_classes']
print(f'Train: {data["X_train"].shape}  Val: {data["X_val"].shape}  Test: {data["X_test"].shape}')
print(f'Features: {n_features}  Classes: {n_classes}')

## 2. Train DNN

In [ ]:
EPOCHS = config['tier2']['training']['epochs']
BATCH_SIZE = config['tier2']['training']['batch_size']
DROPOUT = config['tier2']['training']['dropout_rate']

# -- DNN --
dnn_model = build_dnn(n_features, n_classes, dropout_rate=DROPOUT)
dnn_model.summary()

In [ ]:
dnn_dir = resolve_path('models/tier2')
os.makedirs(dnn_dir, exist_ok=True)

dnn_callbacks = get_training_callbacks(os.path.join(dnn_dir, 'dnn_model.h5'))

dnn_history = dnn_model.fit(
    data['X_train'], data['y_train'],
    validation_data=(data['X_val'], data['y_val']),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=dnn_callbacks,
    verbose=1
)

dnn_val_loss, dnn_val_acc = dnn_model.evaluate(data['X_val'], data['y_val'], verbose=0)
print(f'\nDNN -- Val Loss: {dnn_val_loss:.4f}  Val Accuracy: {dnn_val_acc:.4f}')

## 3. Train CNN

In [ ]:
# Reshape for CNN: (samples, features, 1)
cnn_extractor = FeatureExtractor('CNN', n_features)
X_train_cnn = cnn_extractor.reshape(data['X_train'])
X_val_cnn = cnn_extractor.reshape(data['X_val'])
X_test_cnn = cnn_extractor.reshape(data['X_test'])

cnn_model = build_cnn(cnn_extractor.get_input_shape(), n_classes, dropout_rate=DROPOUT)
cnn_callbacks = get_training_callbacks(os.path.join(dnn_dir, 'cnn_model.h5'))

cnn_history = cnn_model.fit(
    X_train_cnn, data['y_train'],
    validation_data=(X_val_cnn, data['y_val']),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cnn_callbacks,
    verbose=1
)

cnn_val_loss, cnn_val_acc = cnn_model.evaluate(X_val_cnn, data['y_val'], verbose=0)
print(f'\nCNN -- Val Loss: {cnn_val_loss:.4f}  Val Accuracy: {cnn_val_acc:.4f}')

## 4. Train LSTM

In [ ]:
# Reshape for LSTM: (samples, 1, features)
lstm_extractor = FeatureExtractor('LSTM', n_features)
X_train_lstm = lstm_extractor.reshape(data['X_train'])
X_val_lstm = lstm_extractor.reshape(data['X_val'])
X_test_lstm = lstm_extractor.reshape(data['X_test'])

lstm_model = build_lstm(lstm_extractor.get_input_shape(), n_classes, dropout_rate=DROPOUT)
lstm_callbacks = get_training_callbacks(os.path.join(dnn_dir, 'lstm_model.h5'))

lstm_history = lstm_model.fit(
    X_train_lstm, data['y_train'],
    validation_data=(X_val_lstm, data['y_val']),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=lstm_callbacks,
    verbose=1
)

lstm_val_loss, lstm_val_acc = lstm_model.evaluate(X_val_lstm, data['y_val'], verbose=0)
print(f'\nLSTM -- Val Loss: {lstm_val_loss:.4f}  Val Accuracy: {lstm_val_acc:.4f}')

## 5. Compare Validation Accuracy

In [ ]:
results = pd.DataFrame({
    'Model': ['DNN', 'CNN', 'LSTM'],
    'Val Loss': [dnn_val_loss, cnn_val_loss, lstm_val_loss],
    'Val Accuracy': [dnn_val_acc, cnn_val_acc, lstm_val_acc]
})
results = results.sort_values('Val Accuracy', ascending=False)
print(results.to_string(index=False))

best_model_name = results.iloc[0]['Model']
print(f'\nBest model: {best_model_name}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(results['Model'], results['Val Accuracy'], color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Validation Accuracy')
ax.set_title('Tier 2 Model Comparison')
for bar, acc in zip(bars, results['Val Accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Training History Plots

In [ ]:
histories = {
    'DNN': dnn_history.history,
    'CNN': cnn_history.history,
    'LSTM': lstm_history.history
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (name, hist) in enumerate(histories.items()):
    # Loss
    axes[0, idx].plot(hist['loss'], label='Train')
    axes[0, idx].plot(hist['val_loss'], label='Val')
    axes[0, idx].set_title(f'{name} -- Loss')
    axes[0, idx].set_xlabel('Epoch')
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)

    # Accuracy
    axes[1, idx].plot(hist['accuracy'], label='Train')
    axes[1, idx].plot(hist['val_accuracy'], label='Val')
    axes[1, idx].set_title(f'{name} -- Accuracy')
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Training History for All Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

class_names = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R'][:n_classes]

# DNN predictions
dnn_pred = np.argmax(dnn_model.predict(data['X_test'], verbose=0), axis=1)
# CNN predictions
cnn_pred = np.argmax(cnn_model.predict(X_test_cnn, verbose=0), axis=1)
# LSTM predictions
lstm_pred = np.argmax(lstm_model.predict(X_test_lstm, verbose=0), axis=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, preds, name in zip(axes, [dnn_pred, cnn_pred, lstm_pred], ['DNN', 'CNN', 'LSTM']):
    cm = confusion_matrix(data['y_test'], preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name} Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report for the best model
best_preds = {'DNN': dnn_pred, 'CNN': cnn_pred, 'LSTM': lstm_pred}[best_model_name]
print(f'Classification Report -- {best_model_name} (Best Model)')
print('=' * 60)
print(classification_report(data['y_test'], best_preds, target_names=class_names, zero_division=0))

## 8. Save Best Model

In [ ]:
best_model_obj = {'DNN': dnn_model, 'CNN': cnn_model, 'LSTM': lstm_model}[best_model_name]
best_path = resolve_path(config['tier2']['model_path'])
os.makedirs(os.path.dirname(best_path), exist_ok=True)
best_model_obj.save(best_path)
print(f'Best model ({best_model_name}) saved to: {best_path}')

## Summary

- Trained DNN, CNN, and LSTM models on the preprocessed dataset.
- Compared validation accuracy and selected the best performer.
- Visualized training curves and confusion matrices.
- Saved the best model for downstream use in adversarial evaluation.

Next: `04_gan_training.ipynb` -- train the WGAN-GP to generate synthetic attack traffic.